In [14]:
import pandas as pd
import re

1. Load Data

In [15]:
file = "RawMessyData.xlsx"

raw_ing = pd.read_excel(file, sheet_name="Client_Ingredient_Upload")
raw_sup = pd.read_excel(file, sheet_name="Supplier_Reference")

ing = raw_ing.copy()
sup = raw_sup.copy()

2. Column Names

In [16]:
def clean_cols(df):
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"[\s\-]+", "_", regex=True)
                  .str.replace(r"[^\w_]", "", regex=True))
    return df
ing = clean_cols(ing)
sup = clean_cols(sup)
print(ing.columns.tolist())

['ingredient_id', 'ingredient_name', 'inci_name', 'cas_number', 'function', 'concentration_pct', 'supplier', 'supplier_code', 'safety_status', 'date_added', 'notes']


3. String cleaning: whitespace, casing, supplier code bleed.

In [17]:
str_cols = ing.select_dtypes("object").columns
ing[str_cols] = ing[str_cols].apply(lambda s: s.str.strip())
# supplier code bleed: strip leading "XXX-#### " pattern from name
ing["ingredient_name"] = (ing["ingredient_name"]
    .str.replace(r"^[A-Z]{2,5}-\d{3,5}\s+", "", regex=True)
    .str.title())
ing["inci_name"]       = ing["inci_name"].str.upper()
ing["function"]        = ing["function"].str.title()
ing["supplier"]        = ing["supplier"].str.title()
sup["supplier_name"]   = sup["supplier_name"].str.title()
sup["country"]         = sup["country"].str.title()

4. Controlled vocabulary: safety_status typos.

In [18]:
STATUS_MAP = {"approved": "Approved", "aproved": "Approved",
             "restricted": "Restricted", "under review": "Under Review"}
ing["safety_status"] = (ing["safety_status"].str.lower()
    .map(STATUS_MAP)
    .fillna("UNKNOWN — review required"))

5. Data types: concentration_pct.

In [19]:
ing["concentration_pct"] = (
    ing["concentration_pct"].astype(str)
    .str.replace("%", "", regex=False).str.strip()
    .astype(float))

6. Out-of-range values: concentration_pct.

In [20]:
mask_oor = ~ing["concentration_pct"].between(0, 100)
if mask_oor.any():
    print("Out-of-range concentration rows:")
    print(ing.loc[mask_oor, ["ingredient_id", "concentration_pct"]])
ing.loc[mask_oor, "concentration_pct"] = pd.NA

Out-of-range concentration rows:
   ingredient_id  concentration_pct
0        ING-001                NaN
1        ING-002                NaN
2        ING-001                NaN
3        ING-003                NaN
4        ING-004                NaN
5        ING-005                NaN
6        ING-006                NaN
8        ING-008                NaN
9        ING-009                NaN
10       ING-010                NaN
11       ING-011                NaN
12       ING-012                NaN
13       ING-013                NaN
14       ING-014                NaN
15       ING-015                NaN
16       ING-016                NaN
17       ING-017                NaN
18       ING-018                NaN
19       ING-019                NaN
20       ING-020                NaN


7. Date format normalization

In [28]:
ing["date_added"] = pd.to_datetime(
    ing["date_added"], errors="coerce"
).dt.strftime("%Y-%m-%d")

8. Supplier name cross-sheet consistency

In [22]:
SUPPLIER_MAP = {
    "Dsm": "DSM", "Symrise": "Symrise AG",
    "Symrise Ag": "Symrise AG"}
ing["supplier"]      = ing["supplier"].replace(SUPPLIER_MAP)
sup["supplier_name"] = sup["supplier_name"].replace(SUPPLIER_MAP)

9. Missing required fields

In [23]:
REQUIRED = ["ingredient_name", "inci_name", "cas_number"]
missing = ing[ing[REQUIRED].isnull().any(axis=1)]
if not missing.empty:
    print(f"{len(missing)} rows with missing required fields:")
    print(missing[["ingredient_id"] + REQUIRED])

4 rows with missing required fields:
   ingredient_id ingredient_name        inci_name cas_number
4        ING-004             NaN  BUTYLENE GLYCOL   107-88-0
6        ING-006         Retinol          RETINOL        NaN
11       ING-011       Fragrance           PARFUM        NaN
14       ING-014             NaN              NaN        NaN


10. Duplicates

In [24]:
n_exact = ing.duplicated().sum()
print(f"Exact duplicates: {n_exact}")
ing = ing.drop_duplicates()
# near-duplicates by ingredient_id (casing already fixed above)
near_dups = ing[ing.duplicated("ingredient_id", keep=False)]
if not near_dups.empty:
    print("Near-duplicate ingredient_ids:")
    print(near_dups[["ingredient_id", "ingredient_name"]])

Exact duplicates: 1


11. Final validation summary

In [25]:
print("=== Cleaned ingredients ===")
print(ing.dtypes)
print("\nNull counts:")
print(ing.isnull().sum())
print(f"\nRows: {len(ing)} | Cols: {len(ing.columns)}")
ing.head(10)

=== Cleaned ingredients ===
ingredient_id         object
ingredient_name       object
inci_name             object
cas_number            object
function              object
concentration_pct    float64
supplier              object
supplier_code         object
safety_status         object
date_added            object
notes                 object
dtype: object

Null counts:
ingredient_id         0
ingredient_name       2
inci_name             1
cas_number            3
function              1
concentration_pct    19
supplier              2
supplier_code         1
safety_status         0
date_added            2
notes                14
dtype: int64

Rows: 20 | Cols: 11


,ingredient_id,ingredient_name,inci_name,cas_number,function,concentration_pct,supplier,supplier_code,safety_status,date_added,notes
0,ING-001,Water,AQUA,7732-18-5,Solvent,NaN,Brenntag,BRN-4421,Approved,2024-01-15,NaN
1,ING-002,Glycerin,GLYCERIN,56-81-5,Humectant,NaN,Univar Solutions,UNI-8832,Approved,2024-01-15,NaN
3,ING-003,Niacinamide,NIACINAMIDE,98-92-0,Skin Conditioning,NaN,DSM,DSM-0091,Approved,2024-01-20,NaN
4,ING-004,NaN,BUTYLENE GLYCOL,107-88-0,Humectant,NaN,NaN,ASHLAND-221,Approved,2024-01-22,NaN
5,ING-005,Hyaluronic Acid,SODIUM HYALURONATE,9067-32-7,Humectant,NaN,Bloomage Biotech,BB-5510,Approved,2024-01-25,NaN
6,ING-006,Retinol,RETINOL,NaN,Skin Conditioning,NaN,Basf,BASF-7741,Restricted,2024-02-01,Requires stability testing
7,ING-007,Cetyl Alcohol,CETYL ALCOHOL,36653-82-4,Emollient,2.0,Croda,CRD-3302,Approved,2024-02-03,NaN
8,ING-008,Water,AQUA,7732-18-5,Solvent,NaN,Brenntag,BRN-4421,Approved,2024-01-15,NaN
9,ING-009,Phenoxyethanol,PHENOXYETHANOL,122-99-6,Preservative,NaN,Symrise AG,SYM-1190,Approved,2024-02-05,NaN
10,ING-010,Phenoxyethanol,PHENOXYETHANOL,122-99-6,Preservative,NaN,Symrise AG,SYM-1190,Approved,2024-02-05,same as ING-009?


12. Export

In [27]:
with pd.ExcelWriter("ingredients_CLEAN.xlsx", engine="openpyxl") as writer:
    ing.to_excel(writer, sheet_name="Client_Ingredient_Upload", index=False)
    sup.to_excel(writer, sheet_name="Supplier_Reference", index=False)
print("Saved: MessyData_CLEAN.xlsx")

Saved: MessyData_CLEAN.xlsx


In [29]:
import os
print(os.getcwd())


/Users/lucyconsidine/MessySpreadsheetSample
